# 💳 Project Task: GoPay Fintech Analytics
## Data Cleaning, Feature Engineering & Exploratory Data Analysis

**Module 2 — Python for Data Analysis | Purwadhika Digital Technology School**

---

> ⚠️ **Jangan di-run dulu.** Copy notebook ini terlebih dahulu, baru kerjakan di file copy-an kamu.

---

### Konteks Bisnis
GoPay telah berkembang menjadi tulang punggung ekosistem Super App GoTo. Fitur GoPayLater — layanan kredit berbasis limit — berhasil mendongkrak GTV, namun kini menghadapi masalah serius: tingkat NPL (Non-Performing Loan) yang meningkat, bug validasi limit kredit, dan inkonsistensi data dari puluhan micro-service.

Kamu berperan sebagai Data Analyst di tim **Risk Management GoPay** yang diminta untuk membersihkan data, mengidentifikasi pola gagal bayar, dan memberikan rekomendasi perbaikan credit scoring.

**Dataset (3 tabel):**
- `gopay_users.csv` — 35.000 baris (Dimensi User)
- `gopay_services.csv` — 20 baris (Dimensi Layanan)
- `gopay_transactions.csv` — 300.000 baris (Fakta Transaksi)

---

### ⚠️ Catatan Penting
- Task ini **open-ended** — tidak ada satu jawaban yang mutlak benar
- Yang dinilai: **ketepatan keputusan**, **kualitas justifikasi**, dan **kedalaman analisis**
- Setiap keputusan di Data Cleaning & Feature Engineering **wajib disertai penjelasan** di markdown cell
- EDA dikerjakan **tanpa visualisasi** — gunakan pandas aggregation, filtering, sorting, dan merge

---

### 🚨 Business Context Error (Wajib Diinvestigasi)
> Lebih dari **50% transaksi GoPayLater** memiliki `amount` yang **melebihi `paylater_limit`** user yang bersangkutan.  
> Ini adalah bug sistematis pada validasi limit kredit — bukan sekadar outlier biasa.  
> Identifikasi, kuantifikasi dampak finansialnya, dan rekomendasikan perbaikan.

---
## 0. Import & Load Data

In [68]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

In [69]:
# Load semua dataset
# Sesuaikan path dengan lokasi file kamu
df_users_raw    = pd.read_csv('gopay_users.csv')
df_services_raw = pd.read_csv('gopay_services.csv')
df_trx_raw      = pd.read_csv('gopay_transactions.csv')

# Buat copy untuk dikerjakan
df_users    = df_users_raw.copy()
df_services = df_services_raw.copy()
df_trx      = df_trx_raw.copy()


print(f'users       : {df_users.shape}')
print(f'services    : {df_services.shape}')
print(f'transactions: {df_trx.shape}')

users       : (35000, 5)
services    : (20, 3)
transactions: (300000, 8)


---
## 2. Data Cleaning

### 2.1 Eksplorasi Awal (Wajib)

Lakukan eksplorasi menyeluruh pada **ketiga tabel** sebelum membersihkan data apapun.

In [70]:
# Shape dan info umum — lakukan untuk ketiga tabel
# df_users.info()
# df_services.info()
# df_trx.info()

df_users.shape # 35000,5
df_services.shape # 20,3
df_trx.shape # 300000, 8


(300000, 8)

In [71]:
# Tipe data seluruh kolom
print("tipe data seluruh kolom df_users:\n",df_users.dtypes)
print("tipe data seluruh kolom df_services:\n",df_services.dtypes)
print("tipe data seluruh kolom df_trx:\n",df_trx.dtypes)


tipe data seluruh kolom df_users:
 user_id                   object
join_date                 object
gopay_tier                object
internal_credit_score    float64
paylater_limit             int64
dtype: object
tipe data seluruh kolom df_services:
 service_id      object
service_name    object
category        object
dtype: object
tipe data seluruh kolom df_trx:
 trx_id             object
user_id            object
service_id         object
trx_date           object
payment_method     object
amount              int64
late_fee          float64
payment_status     object
dtype: object


In [72]:
# Missing values: jumlah dan persentase per kolom, per tabel
# Tampilkan hanya kolom yang memiliki missing values
na_values=df_users.isnull().sum()
print("Total nilai kosong tiap kolom df_user:")
na_values.apply(lambda x: f"{x} - {x/len(df_users):.2%}")

# na_values=df_services.isnull().sum()
# print("Total nilai kosong tiap kolom df_user:")
# na_values.apply(lambda x: f"{x} - {x/len(df_services):.2%}")


# na_values=df_trx.isnull().sum()
# print("Total nilai kosong tiap kolom df_user:")
# na_values.apply(lambda x: f"{x} - {x/len(df_trx):.2%}")

#display(df_users["internal_credit_score"].value_counts())


Total nilai kosong tiap kolom df_user:


user_id                      0 - 0.00%
join_date                    0 - 0.00%
gopay_tier                   0 - 0.00%
internal_credit_score    5250 - 15.00%
paylater_limit               0 - 0.00%
dtype: object

In [73]:
# Distribusi kolom-kolom kritis
# payment_method, amount, late_fee, payment_status, gopay_tier, internal_credit_score
df_kritis = pd.merge(df_trx[['user_id','payment_method','amount','late_fee']],
                     df_users[['user_id','gopay_tier','internal_credit_score']], on='user_id', how='left')
# df_kritis['payment_method'].value_counts()
# df_kritis['amount'].value_counts()
# df_kritis['late_fee'].value_counts()
# df_kritis['gopay_tier'].value_counts()

In [74]:
# Cek konsistensi relasi antar tabel
# Apakah semua service_id di transactions ada di services?
trx_u= df_trx["service_id"].unique()
srv_u=df_services["service_id"].unique()

comparison_ts = df_trx["service_id"].isin(df_services["service_id"].unique())
comparison_ts.unique()
# Ya, semua service id transaction ada di services

# Apakah semua user_id di transactions ada di users?

user_compare = df_trx["user_id"].isin(df_users["user_id"].unique())
display(user_compare.unique())


array([ True])

**✍️ Ringkasan Temuan Eksplorasi:**

Kami menemukan ada missing value di:
data user sebanyak 15%(5250), kolom: internal_credit_score

Kami juga menemukan inkonsistensi di:
df transaction:
kolom late_fee: ada nominal negative (-15000, 75 row, 999999 di 76 row)
kolom payment method: ada beragam penulisan GoPay, gopay, PayLater, GoPayLater, Cash, CASH yang perlu distandarisasi.

> 

---
### 2.2 Kerangka Identifikasi Missing Values

Sebelum menangani missing values pada kolom manapun, identifikasi dulu **jenis missing value-nya**.

| Jenis | Definisi Singkat | Implikasi Penanganan | Contoh di Dataset Ini |
|---|---|---|---|
| **MCAR** *(Missing Completely At Random)* | Nilai kosong tidak berkaitan dengan kolom lain. Pola missing benar-benar acak. | Relatif aman di-impute atau di-drop tanpa bias signifikan. | Sebagian kecil `internal_credit_score` kosong karena gangguan sistem scraping acak. |
| **MAR** *(Missing At Random)* | Nilai kosong berkaitan dengan kolom **lain**, bukan dengan nilai kolom itu sendiri. | Imputation berbasis kolom lain lebih tepat. Drop bisa menyebabkan bias. | `internal_credit_score` kosong mungkin berkorelasi dengan `gopay_tier` atau `join_date`. |
| **MNAR** *(Missing Not At Random)* | Nilai kosong berkaitan langsung dengan nilai yang seharusnya ada. Ada alasan sistematis. | Imputation apapun berisiko misleading. Perlu keputusan bisnis eksplisit. | `internal_credit_score` kosong justru karena user tidak pernah bertransaksi — nilai kosong itu sendiri adalah sinyal risiko. |

> 💡 **Cara Menggunakan Kerangka Ini:**
> Untuk setiap kolom bermasalah, tanyakan:
> 1. Apakah pola missing-nya acak, atau ada pola tertentu?
> 2. Apakah nilai kosong berkaitan dengan kolom lain?
> 3. Apakah nilai kosong itu sendiri mengandung informasi bisnis?
>
> Justifikasi reasoning kamu lebih penting dari labelnya.

In [75]:
display(df_users)

,user_id,join_date,gopay_tier,internal_credit_score,paylater_limit
0,GP-000001,2022-05-26,Basic,NaN,0
1,GP-000002,2023-07-12,Plus,751.00,500000
2,GP-000003,2022-05-09,Plus,728.00,1500000
3,GP-000004,2021-08-28,Basic,394.00,0
4,GP-000005,2021-05-31,Basic,333.00,0
...,...,...,...,...,...
34995,GP-034996,2022-03-12,Plus,544.00,1500000
34996,GP-034997,2023-07-23,Plus,NaN,5000000
34997,GP-034998,2023-08-06,Plus,665.00,1500000
34998,GP-034999,2022-05-29,Basic,391.00,0


---
### 2.3 Penanganan Kolom `payment_method`

Kolom ini memiliki 8 varian penulisan untuk 3 metode pembayaran yang berbeda, akibat inkonsistensi penamaan antar micro-service.

| Varian Asli | Metode Sebenarnya |
|---|---|
| `GoPay`, `gopay`, `GO-PAY` | GoPay (saldo digital) |
| `GoPayLater`, `PayLater`, `gopay_later` | GoPayLater (kredit) |
| `Cash`, `CASH` | Cash |

> 🧠 **Critical Thinking Prompt:**  
> Setelah standarisasi, periksa ulang: apakah ada user **Basic tier** yang menggunakan GoPayLater?  
> Secara aturan bisnis, PayLater hanya boleh digunakan oleh user Plus.  
> Jika ada, apakah itu error data atau bug sistem validasi?

In [76]:
# Lihat semua nilai unik di payment_method beserta frekuensinya
df_kritis['payment_method'].value_counts()

payment_method
GoPay          90240
GoPayLater     60014
gopay          59854
PayLater       30017
GO-PAY         15136
Cash           14978
CASH           14972
gopay_later    14789
Name: count, dtype: int64

**✍️ Mapping standarisasi yang kamu buat:**
- Varian asli → nilai standar (tuliskan mapping lengkapnya):
- Format standar yang kamu pilih dan alasannya:
- Temuan setelah standarisasi (apakah ada Basic tier yang pakai GoPayLater?):

> 

In [77]:
# TODO: Standarisasi payment_method
# Simpan hasil ke kolom baru: payment_method_clean
df_trx["payment_method_clean"] = df_trx["payment_method"].replace({
    "GoPay": "GoPay",
    "gopay": "GoPay",
    "GO-PAY": "GoPay",
    "PayLater": "PayLater",
    "GoPayLater": "PayLater",
    "gopay_later": "PayLater",
    "Cash": "Cash",
    "CASH": "Cash"
})

In [79]:
# Verifikasi: cek Basic tier yang menggunakan GoPayLater setelah standarisasi
tier_paylater = df_kritis[df_kritis['payment_method'] == 'GoPayLater']['gopay_tier'].value_counts()
print(tier_paylater)

gopay_tier
Plus     36138
Basic    23876
Name: count, dtype: int64


---
### 2.4 Penanganan `internal_credit_score`

Kolom `internal_credit_score` di tabel users memiliki **5.250 nilai kosong (~15%)** — variabel kritis untuk analisis risiko kredit.

> 🧠 **Critical Thinking Prompt:**  
> Apakah nilai kosong ini karena sistem gagal mencatat, atau karena user memang belum punya histori kredit?  
> User tanpa credit score = *unscored* — di industri fintech, ini dianggap risiko tersendiri.  
> Keputusan kamu di sini akan langsung mempengaruhi hasil analisis profil risiko di Section 4.

In [ ]:
# Investigasi pola missing values pada internal_credit_score
# Apakah berkorelasi dengan gopay_tier, paylater_limit, atau join_date?


In [ ]:
# Cek: apakah user yang credit_score-nya missing lebih banyak yang default?
# Hint: merge dengan df_trx, lalu bandingkan default rate


**✍️ Analisis & Justifikasi:**
- **Jenis missing value (MCAR / MAR / MNAR):** dan alasan klasifikasi kamu:
- Temuan investigasi pola missing (berkorelasi dengan tier? join_date?):
- Apakah user tanpa credit score memiliki default rate yang berbeda?
- Keputusan penanganan (drop / impute / pertahankan NaN) dan alasan:

> 

In [ ]:
# TODO: Implementasi penanganan missing values internal_credit_score


---
### 2.5 Penanganan `late_fee`

Kolom `late_fee` memiliki dua jenis anomali yang berbeda sifatnya — tangani secara terpisah.

> 🧠 **Critical Thinking Prompt:**  
> Di industri fintech, denda keterlambatan diatur oleh regulasi OJK.  
> Nilai `late_fee` yang sangat besar bisa berarti bug sistem, bukan kebijakan yang valid.  
> Keputusan kamu harus mempertimbangkan aspek **compliance**, bukan hanya statistik.

In [ ]:
# Investigasi distribusi late_fee secara menyeluruh
# Berapa nilai negatif? Berapa nilai ekstrem?


In [ ]:
# Anomali 1: Nilai negatif
# Apakah terjadi pada payment_status tertentu?


In [ ]:
# Anomali 2: Nilai ekstrem tinggi (> Rp 10 juta)
# Apakah ada pola pada service atau user tertentu?


**✍️ Analisis & Justifikasi — Anomali 1 (late_fee negatif):**
- Jumlah baris terdampak:
- Hipotesis penyebab (logical error? refund denda yang salah catat?):
- Keputusan penanganan dan alasan:

> 

**✍️ Analisis & Justifikasi — Anomali 2 (late_fee ekstrem):**
- Jumlah baris terdampak dan range nilainya:
- Threshold yang kamu pilih untuk mendefinisikan 'ekstrem' dan alasannya:
- Hipotesis penyebab (bug sistem? kebijakan tidak terkontrol?):
- Keputusan penanganan (cap / drop / flag) dan alasan:

> 

In [ ]:
# TODO: Implementasi penanganan Anomali 1 (late_fee negatif)


In [ ]:
# TODO: Implementasi penanganan Anomali 2 (late_fee ekstrem)


---
### 2.6 Penanganan Anomali Tanggal: `trx_date` sebelum `join_date`

Terdapat **~30.177 transaksi (~10%)** dengan `trx_date` lebih awal dari `join_date` user — secara logika bisnis tidak mungkin terjadi.

> 🧠 **Critical Thinking Prompt:**  
> Di konteks fintech, transaksi sebelum akun dibuat bisa mengindikasikan **fraud** atau **data migration issue**.  
> Drop vs. flag memiliki implikasi berbeda: drop menghilangkan sinyal fraud, flag mempertahankannya untuk analisis.  
> Apakah anomali ini lebih banyak terjadi pada user yang akhirnya **Default**?

In [86]:
# Konversi kolom tanggal ke datetime
df_trx['trx_date'] = pd.to_datetime(df_trx['trx_date'])
df_users['join_date'] = pd.to_datetime(df_users['join_date'])


In [88]:
# Identifikasi transaksi dengan trx_date < join_date
# Investigasi: seberapa besar selisih tanggalnya? Distribusi selisih negatif?
df_merged = df_trx.merge(df_users, on='user_id', how='left')

# hitung selisih
df_merged['date_diff_days'] = (df_merged['trx_date'] - df_merged['join_date']).dt.days
df_merged['date_diff_days'].value_counts() # selisih tanggal mulai dari 160days sampai 720days

# cari data anomali
df_anomali = df_merged[df_merged['trx_date'] < df_merged['join_date']] # get yang less than aja
df_merged['is_anomali_tanggal'] = df_merged['date_diff_days'] < 0 # get yang negatif
#print(df_anomali[['trx_id', 'user_id', 'trx_date', 'join_date', 'date_diff_days']])

print(df_anomali['date_diff_days'].describe())
# Hasil
# count   30177.00
# mean      -89.90
# std        63.30
# min      -269.00
# 25%      -135.00
# 50%       -79.00
# 75%       -37.00
# max        -1.00

count   30177.00
mean      -89.90
std        63.30
min      -269.00
25%      -135.00
50%       -79.00
75%       -37.00
max        -1.00
Name: date_diff_days, dtype: float64


In [112]:
# Apakah anomali ini berkorelasi dengan payment_status = Default?
# Apakah tersebar merata atau terkonsentrasi pada user/tanggal tertentu?
from scipy.stats import chi2_contingency
# cek korelasi:
crosstab_status1 = pd.crosstab(
    df_merged['is_anomali_tanggal'],
    df_merged['payment_status'],
    normalize='index'
)
crosstab_status2 = pd.crosstab(
    df_merged['is_anomali_tanggal'],
    df_merged['payment_method_clean'],
    normalize='index'
)

crosstab_status3 = pd.crosstab(
    df_merged['is_anomali_tanggal'],
    df_merged['user_id'],
    normalize='index'
)
crosstab_raw = pd.crosstab(
    df_merged['is_anomali_tanggal'], 
    df_merged['trx_date']
)

# 2. Jalankan Uji Chi-Square
chi2, p_value, dof, expected = chi2_contingency(crosstab_status1)

print(f"P-Value: {p_value}")
print(crosstab_status1)

#cek anomali per bulan
anomali_per_bulan = df_anomali['trx_date'].dt.to_period('M').value_counts().sort_index()
print(anomali_per_bulan)
# df_merged
# Ditemukan: tidak terdapat korelasi dengan payment status, user id maupun trx_date tertentu
# Tapi ada sedikit pola per bulan, anomali konsisten menurun, Januari - September, 6000 - 200an

P-Value: 0.9999910782883896
payment_status      Default  Paid  Pending
is_anomali_tanggal                        
False                  0.05  0.91     0.04
True                   0.05  0.91     0.03
trx_date
2023-01    6709
2023-02    5211
2023-03    4937
2023-04    4193
2023-05    3381
2023-06    2512
2023-07    1859
2023-08    1106
2023-09     269
Freq: M, Name: count, dtype: int64


**✍️ Analisis & Justifikasi:**
- Jumlah baris terdampak dan distribusi selisih tanggal:30,177 baris terdampak anomali
- **Jenis anomali (acak / berpola):** Anomali berpola per bulan dari januari 2023 di 6000 transaksi menurun di september 2023 pada 200an transaksi
- Hipotesis penyebab (migration error? clock skew? fraud?): Migration error. karena menurun secara sistematis. dan tidak ada korelasi pada user/payment status
- Apakah anomali ini berkorelasi dengan Default? Implikasi untuk analisis risiko: Tidak berkorelasi, p 0.6 tidak signifikan
- Keputusan penanganan (drop / flag / pertahankan) dan alasan:FLAG. untuk konfirmasi ke tim engineering

> 

In [ ]:
# TODO: Implementasi penanganan anomali tanggal
# tambah kolom is_anomali_tanggal untuk FLAG (bool)
#df_trx['is_anomali_tanggal'] = df_merged['date_diff_days'] < 0 # get yang negatif

df_trx


,trx_id,user_id,service_id,trx_date,payment_method,amount,late_fee,payment_status,payment_method_clean,is_anomali_tanggal
0,GTRX-0000001,GP-018595,SVC-014,2023-03-10 06:00:00,GoPay,157588,0.00,Paid,GoPay,False
1,GTRX-0000002,GP-000244,SVC-014,2023-12-12 04:00:00,GoPay,594659,0.00,Paid,GoPay,False
2,GTRX-0000003,GP-006568,SVC-001,2023-11-30 06:00:00,gopay,34229,0.00,Paid,GoPay,False
3,GTRX-0000004,GP-001920,SVC-001,2023-06-12 14:00:00,gopay,234183,0.00,Paid,GoPay,False
4,GTRX-0000005,GP-030810,SVC-002,2023-11-01 20:00:00,PayLater,380431,0.00,Paid,PayLater,False
...,...,...,...,...,...,...,...,...,...,...
299995,GTRX-0299996,GP-024329,SVC-002,2023-12-26 20:00:00,CASH,327414,0.00,Paid,Cash,False
299996,GTRX-0299997,GP-008218,SVC-019,2023-02-03 10:00:00,GoPay,17764,0.00,Paid,GoPay,False
299997,GTRX-0299998,GP-026202,SVC-005,2023-01-29 03:00:00,GoPayLater,253066,0.00,Paid,PayLater,False
299998,GTRX-0299999,GP-000398,SVC-015,2023-06-09 16:00:00,GoPay,502706,0.00,Paid,GoPay,False


---
### 2.7 Penanganan Business Logic Error: Transaksi PayLater Melebihi Limit

**Ini adalah anomali paling kritis di dataset ini.** Lebih dari 50% transaksi GoPayLater memiliki `amount` yang melebihi `paylater_limit` user, termasuk user Basic tier yang seharusnya tidak punya PayLater sama sekali.

| Tipe Pelanggaran | Deskripsi |
|---|---|
| **Basic tier pakai PayLater** | User dengan `paylater_limit = 0` bertransaksi dengan GoPayLater |
| **Plus tier melebihi limit** | User PayLater sah, tapi `amount > paylater_limit` |
| **Transaksi valid** | User Plus dengan `amount ≤ paylater_limit` |

> 🧠 **Critical Thinking Prompt:**  
> Jangan drop transaksi over-limit — ini adalah **data paling berharga** untuk memahami bug dan pola default.  
> Pertahankan dengan flag, lalu analisis secara terpisah.  
> **Dropping = menghilangkan bukti.**

In [ ]:
# Merge transaksi GoPayLater dengan data paylater_limit user
# Identifikasi tipe pelanggaran untuk setiap transaksi


In [ ]:
# Kuantifikasi: berapa jumlah dan total nilai (Rupiah) dari setiap tipe pelanggaran?


In [ ]:
# Kritis: apakah transaksi over-limit berkorelasi dengan payment_status = Default?
# Bandingkan default rate antara: transaksi valid vs over-limit


**✍️ Analisis & Justifikasi:**
- Jumlah dan persentase setiap tipe pelanggaran:
- Total nilai Rupiah yang terlibat dalam pelanggaran:
- Apakah over-limit berkorelasi dengan Default? Temuan kamu:
- Keputusan penanganan (flag, bukan drop) dan kolom flag yang kamu buat:
- Hipotesis mengapa bug ini bisa terjadi di sistem:

> 

In [ ]:
# TODO: Buat kolom flag untuk tipe pelanggaran PayLater
# Contoh: 'valid', 'over_limit', 'unauthorized'


---
### 2.8 Penanganan Duplikat & Integritas Data

In [ ]:
# 1. Cek exact duplicates di setiap tabel


In [ ]:
# 2. Cek duplikat trx_id


In [ ]:
# 3. Cek service_id di transactions yang tidak ada di services


In [ ]:
# 4. Cek inkonsistensi logika: payment_status = Pending tapi late_fee > 0


**✍️ Analisis & Justifikasi:**
- Masalah yang ditemukan dan jumlah baris terdampak:
- Hipotesis untuk setiap masalah:
- Keputusan penanganan per masalah:

> 

In [ ]:
# TODO: Implementasi keputusan penanganan masalah integritas


---
## 3. Feature Engineering

### 3.1 Fitur Wajib

Buat 8 kolom berikut. Sertakan penjelasan singkat business value-nya di setiap fitur.

#### ⚙️ `payment_method_clean`
*(Sudah dibuat di Section 2.3 — pastikan sudah ada di df_trx)*

In [115]:
df_trx["payment_method_clean"]

0            GoPay
1            GoPay
2            GoPay
3            GoPay
4         PayLater
            ...   
299995        Cash
299996       GoPay
299997    PayLater
299998       GoPay
299999       GoPay
Name: payment_method_clean, Length: 300000, dtype: object

#### ⚙️ `credit_score_tier`

> 💡 Default threshold: Poor (300–499), Fair (500–649), Good (650–749), Excellent (750–850).  
> Sesuaikan jika analisis distribusi kamu menunjukkan pembagian yang lebih bermakna secara bisnis.

**✍️ Threshold yang kamu pilih dan alasannya:**

> 

In [ ]:
# TODO: Buat credit_score_tier di df_users
# Pertimbangkan: bagaimana menangani user yang credit_score-nya NaN? 0
credit_tier=[]
for score in df_users["internal_credit_score"]:
    if pd.isna(score):
        credit_tier.append("None") 
    elif score < 500:
        credit_tier.append("Poor")
    elif score < 650:
        credit_tier.append("Fair")
    elif score < 750:
        credit_tier.append("Good")
    else:
        credit_tier.append("Excellent")

#df_users["credit_score_tier"] = credit_tier # di run 1x saja
df_users.head(3)

,user_id,join_date,gopay_tier,internal_credit_score,paylater_limit,credit_score_tier
0,GP-000001,2022-05-26,Basic,NaN,0,0
1,GP-000002,2023-07-12,Plus,751.00,500000,Excellent
2,GP-000003,2022-05-09,Plus,728.00,1500000,Good


#### ⚙️ `is_paylater_violation`

> 💡 Perlu merge df_trx dengan df_users untuk mendapatkan paylater_limit per transaksi.

In [ ]:
# TODO: Buat is_paylater_violation (boolean)
# True jika payment_method_clean == 'gopaylater' AND amount > paylater_limit


#### ⚙️ `paylater_usage_ratio`

> 💡 Hanya relevan untuk transaksi GoPayLater. Untuk transaksi non-PayLater, isi dengan NaN.

In [ ]:
# TODO: Buat paylater_usage_ratio (amount / paylater_limit)
# Handle division by zero untuk user dengan paylater_limit = 0


#### ⚙️ `user_tenure_days`

> 💡 Tentukan sendiri tanggal referensi yang kamu gunakan dan justifikasikan.

**✍️ Tanggal referensi yang kamu gunakan dan alasannya:**

> 

In [ ]:
# TODO: Buat user_tenure_days di df_users


#### ⚙️ `has_late_fee`

In [ ]:
# TODO: Buat has_late_fee (boolean: True jika late_fee > 0)
# Pastikan menggunakan late_fee yang sudah di-clean dari Section 2.5


#### ⚙️ `is_default`

In [ ]:
# TODO: Buat is_default (boolean: True jika payment_status == 'Default')


#### ⚙️ `service_category`

> 💡 Join df_trx dengan df_services untuk mendapatkan kategori layanan per transaksi.

In [ ]:
# TODO: Buat service_category dengan merge ke df_services


---
### 3.2 Fitur Pilihan (Minimal 2)

Pilih minimal 2 dari: `default_rate_per_user`, `avg_amount_per_service_category`, `is_high_risk_transaction`, `credit_utilization_band`, atau fitur buatan sendiri.

#### ⚙️ Fitur Pilihan 1: [Isi nama fitur]

**✍️ Business value dari fitur ini:**

> 

In [ ]:
# TODO: Implementasi Fitur Pilihan 1


#### ⚙️ Fitur Pilihan 2: [Isi nama fitur]

**✍️ Business value dari fitur ini:**

> 

In [ ]:
# TODO: Implementasi Fitur Pilihan 2


---
## 4. Exploratory Data Analysis

> **Aturan:** Semua analisis menggunakan pandas — tanpa visualisasi.  
> Gunakan `.groupby()`, `.agg()`, `.value_counts()`, filtering, sorting, dan **merge antar tabel** saat dibutuhkan.  
> Setiap jawaban **wajib disertai insight** di markdown cell yang tersedia.

---
### 4.1 Analisis Transaksi & Metode Pembayaran

**Soal 1:** Berapa total GTV (Gross Transaction Value) keseluruhan? Breakdown GTV per `payment_method_clean`. Metode mana yang paling dominan dan apa implikasi bisnisnya?

In [ ]:
# Soal 1


**✍️ Insight:**

> 

**Soal 2:** Berapa distribusi `payment_status` secara keseluruhan? Kemudian breakdown **default rate** per `payment_method_clean`. Apakah GoPayLater memiliki default rate yang lebih tinggi?

In [ ]:
# Soal 2


**✍️ Insight:**

> 

**Soal 3:** Berapa rata-rata, median, dan standar deviasi `amount` per `payment_method_clean`? Apa yang bisa disimpulkan dari perbedaan mean vs median?

In [ ]:
# Soal 3


**✍️ Insight:**

> 

**Soal 4:** Analisis `late_fee`: Berapa persentase transaksi yang dikenakan denda? Berapa total `late_fee` yang terkumpul? Breakdown per `payment_method_clean`.

In [ ]:
# Soal 4


**✍️ Insight:**

> 

---
### 4.2 Analisis Risiko Kredit & Profil User

**Soal 5:** Berapa distribusi `credit_score_tier`? Kemudian bandingkan **default rate** (untuk transaksi GoPayLater) antar `credit_score_tier`. Apakah user dengan credit score rendah memiliki default rate yang lebih tinggi?

In [ ]:
# Soal 5
# Hint: merge df_trx (filter GoPayLater) dengan df_users, lalu groupby credit_score_tier


**✍️ Insight:**

> 

**Soal 6:** Berapa persentase `is_paylater_violation = True`? Breakdown antara: Basic tier pakai PayLater vs Plus tier melebihi limit. Berapa total nilai Rupiah yang terlibat?

In [ ]:
# Soal 6


**✍️ Insight:**

> 

**Soal 7:** Apakah ada korelasi antara `paylater_usage_ratio` dan `is_default`? Bandingkan rata-rata `paylater_usage_ratio` antara transaksi yang Default vs yang tidak.

In [ ]:
# Soal 7


**✍️ Insight:**

> 

**Soal 8:** Berapa distribusi `gopay_tier` di antara user yang pernah Default? Apakah user Basic yang 'membobol' sistem PayLater memiliki default rate lebih tinggi dari user Plus yang sah?

In [ ]:
# Soal 8


**✍️ Insight:**

> 

---
### 4.3 Analisis Layanan & Kategori

**Soal 9:** Berapa total GTV dan jumlah transaksi per `service_category`? Kategori mana yang paling tinggi volumenya?

In [ ]:
# Soal 9


**✍️ Insight:**

> 

**Soal 10:** Berapa **default rate** per `service_category` untuk transaksi GoPayLater? Layanan mana yang paling berisiko untuk dibayar dengan PayLater?

In [ ]:
# Soal 10


**✍️ Insight:**

> 

**Soal 11:** Top 5 `service_name` berdasarkan total `late_fee` yang dikumpulkan. Apakah ini mengindikasikan layanan tertentu lebih sering mengalami keterlambatan pembayaran?

In [ ]:
# Soal 11


**✍️ Insight:**

> 

---
### 4.4 Analisis Sistem & Deteksi Anomali *(Implicit — Business Sense Required)*

> Kamu diminta tim **Risk & Compliance** untuk menyusun laporan investigasi sistem PayLater.  
> Temuan ini akan digunakan untuk: (a) menentukan apakah PayLater perlu di-suspend sementara,  
> (b) mengidentifikasi user yang perlu limit adjustment, dan  
> (c) mengestimasi **total kerugian potensial** dari bug yang ada.

Pilih minimal **2 angle analisis** yang paling relevan untuk menjawab kebutuhan investigasi tersebut.

#### 🔍 Investigasi — Angle 1: [Isi judul]

**✍️ Mengapa kamu memilih angle ini untuk investigasi sistem?**

> 

In [ ]:
# Angle 1


**✍️ Insight & Rekomendasi untuk Tim Risk & Compliance:**

> 

#### 🔍 Investigasi — Angle 2: [Isi judul]

**✍️ Mengapa kamu memilih angle ini?**

> 

In [ ]:
# Angle 2


**✍️ Insight & Rekomendasi:**

> 

---
### 4.5 Credit Risk Profiling *(Implicit — Open Ended)*

> Kamu diminta **Chief Risk Officer GoPay** untuk menyusun rekomendasi perbaikan algoritma credit scoring.  
> Tujuan: menentukan kriteria yang lebih ketat untuk pemberian limit PayLater,  
> sehingga NPL bisa ditekan tanpa terlalu banyak membatasi user yang sebenarnya *creditworthy*.

> 🧠 **Critical Thinking Prompt:**  
> Apakah user dengan credit score rendah **selalu** berisiko?  
> Bagaimana dengan user baru yang belum punya credit score sama sekali?  
> Temukan **sweet spot** antara risk mitigation dan business growth.

**Ekspektasi minimal:**
- Minimal 3 variabel/fitur berbeda yang kamu identifikasi sebagai prediktor default yang signifikan
- Profil 'high-risk user' berdasarkan kombinasi variabel tersebut
- Minimal 1 rekomendasi konkret untuk kebijakan limit PayLater yang berbasis data

**✍️ Definisi 'high-risk user' menurut kamu (dalam konteks kredit GoPay):**

> 

#### 📊 Prediktor Default 1: [Nama Variabel]

In [ ]:
# Prediktor 1


#### 📊 Prediktor Default 2: [Nama Variabel]

In [ ]:
# Prediktor 2


#### 📊 Prediktor Default 3: [Nama Variabel]

In [ ]:
# Prediktor 3


#### 🎯 Profil High-Risk User vs Average User

In [ ]:
# Bandingkan karakteristik high-risk user vs keseluruhan user PayLater


**✍️ Rekomendasi Kebijakan Limit PayLater untuk Chief Risk Officer:**

> 

---
## 5. Export Clean Dataset

In [ ]:
# Gabungkan ketiga tabel menjadi satu dataframe final
# Gunakan LEFT JOIN dengan df_trx sebagai tabel utama
# Sertakan semua fitur baru yang telah dibuat

# TODO: Implementasi JOIN
# df_final = df_trx.merge(df_users[...], on='user_id', how='left')
#                  .merge(df_services[...], on='service_id', how='left')

# Export
# df_final.to_csv('gopay_clean.csv', index=False)
# print(f'Dataset berhasil disimpan: gopay_clean.csv')
# print(f'Shape final: {df_final.shape}')
# print(f'Kolom baru yang ditambahkan: {[c for c in df_final.columns if c not in df_trx_raw.columns]}')


---
## 6. Ringkasan & Refleksi

**Keputusan Data Cleaning yang paling challenging dan mengapa:**

> 

**Temuan paling menarik dari EDA (khususnya terkait risiko kredit):**

> 

**Rekomendasi bisnis utama yang bisa diberikan kepada tim Risk Management GoPay:**

> 